# 

This cell installs the `pub-dialogue` package, which is essential for executing subsequent analyses related to clustering and evaluating the robustness of phrase embeddings derived from public dialogue data. By ensuring that the necessary library is available, it facilitates the implementation of clustering algorithms and the assessment of k-sensitivity, both of which are critical for understanding the nuances of concern and benefit phrases in public discourse.

In [ ]:
# @title Install pub-dialogue package
# Local: run 'pip install -e .' once in the repo root.
# Colab:  installs directly from GitHub.
try:
    import pub_dialogue
except ImportError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/mlatcl/pub-dialogue.git",
    ])
    import pub_dialogue

This notebook cell imports essential libraries and modules required for clustering analysis and visualization of phrase embeddings related to concerns and benefits. By leveraging tools like KMeans for clustering and various visualization libraries, it sets the groundwork for analyzing the structure of public dialogue data, which is crucial for understanding sentiment and thematic trends in discussions surrounding specific topics. This foundational step is vital for ensuring that subsequent analyses yield meaningful insights into how different phrases are grouped and perceived in public discourse.

This cell initializes the necessary libraries and sets the configuration for visualizations in the notebook, which is crucial for subsequent analysis. By importing libraries such as NumPy for numerical operations and Matplotlib for plotting, it establishes the foundation for clustering visualizations and ensures that the graphical output is clear and aesthetically pleasing, thereby enhancing the interpretability of the clustering results related to concern and benefit phrases.

In [ ]:
# @title Import libraries and define configuration

import random
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150


This notebook cell sets up the environment for clustering concern phrases by establishing reproducibility through a fixed random seed and defining key parameters for the embedding and clustering models. It specifies the number of clusters, chunking criteria for text processing, and paths for data management, ensuring that the analysis can handle various document formats effectively. This foundational setup is crucial for producing consistent and reliable embeddings, which are essential for the subsequent clustering and analysis of public dialogue on concerns and benefits.

In [ ]:
# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Model Configuration
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4o-mini"

# Clustering - now 75 clusters for concern phrases (smaller units than paragraphs)
N_CONCERN_CLUSTERS = 75
SOFT_MEMBERSHIP_THRESHOLD = 0.3

# Chunking
MIN_CHUNK_WORDS = 40
MAX_CHUNK_WORDS = 500
MIN_CHUNK_CHARS = 80

# When paragraph-level splitting (the default) fails to produce reasonable
# chunks (because the PDF lacks double-newline paragraph separators),
# fall back to sentence-level splitting and repack into chunks of about
# this many words. v18: addresses an issue identified in v17 where 20
# documents (mostly AI policy reports from 2020+) were silently truncated
# because their full text was treated as a single 500-word paragraph.
SENTENCE_FALLBACK_TARGET_WORDS = 300
SENTENCE_FALLBACK_MIN_PARAGRAPHS = 3  # if Method 1 yields fewer paragraphs, fall back

# Processing
EMBEDDING_BATCH_SIZE = 100
EXTRACTION_BATCH_SIZE = 10  # For concern extraction

# Paths — resolve sensibly for both Colab (/content/...) and local (repo root)
try:
    import google.colab  # noqa: F401
    _REPO_ROOT = Path("/content")
except ImportError:
    # Local: notebook lives in repo root, so relative paths work directly
    _REPO_ROOT = Path(".")

PDF_FOLDER        = _REPO_ROOT / "pdfs"
OUTPUT_FOLDER     = _REPO_ROOT / "outputs"
CHECKPOINT_FOLDER = _REPO_ROOT / "checkpoints"
DATA_FOLDER       = _REPO_ROOT / "data"

# Ensure output directories exist
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FOLDER.mkdir(parents=True, exist_ok=True)

This cell initializes the necessary directories for storing outputs and checkpoints, ensuring that the environment is prepared for subsequent analyses. It also suppresses warnings to maintain a clean output and confirms the configuration settings, including the embedding model, target clusters, and the language model used for extraction, which are crucial for understanding the context of the clustering and robustness analysis that follows. This setup is essential for maintaining an organized workflow and ensuring that the analysis is reproducible and clearly defined.

In [ ]:
for folder in [PDF_FOLDER, OUTPUT_FOLDER, CHECKPOINT_FOLDER]:
    folder.mkdir(exist_ok=True)


import warnings
warnings.filterwarnings('ignore')

print("Configuration loaded")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Target clusters: {N_CONCERN_CLUSTERS}")
print(f"  LLM for extraction: {LLM_MODEL}")

This cell imports essential modules and functions from the `pub_dialogue` library, which provide tools for data access, assessment, and analysis relevant to public dialogue research. By establishing these utilities, the cell sets the foundation for subsequent clustering and sensitivity analysis of concern and benefit phrase embeddings, ensuring that the necessary functions for processing and evaluating the data are readily available for the overall analysis.

In [ ]:
# @title Load pub_dialogue — shared helper functions and access/assess/address modules
try:
    import pub_dialogue.utils as du
    import pub_dialogue.access as access
    import pub_dialogue.assess as assess
    import pub_dialogue.address as address
    from pub_dialogue.utils import (
        show_status, show_complete, show_warning,
        save_checkpoint, load_checkpoint,
        CROSSCUTTING_ENTROPY_THRESHOLD,
        EXTRACTION_PROMPT, BENEFIT_EXTRACTION_PROMPT,
        ExtractionResult, extract_phrases, validate_extraction_cache,
        write_extraction_diagnostics,
        filter_missing_source_text, get_embeddings_batch,
        label_cluster, pretty_label, clusters_to_labels,
        clusters_to_lenses, html_escape,
        normalized_entropy, hhi, topk_share, parse_year, tokenize,
        is_privacy_text, entropy_by_year, ai_fingerprint_over_crosscut,
        run_sensitivity,
        vocabulary_frequency_diagnostic, generate_validation_summary,
        extract_chunks_from_pdf, reset_chunk_stats, get_chunk_stats,
        MIN_CHUNK_WORDS, MIN_CHUNK_CHARS, MAX_CHUNK_WORDS,
        SENTENCE_FALLBACK_TARGET_WORDS, SENTENCE_FALLBACK_MIN_PARAGRAPHS,
    )
    print("pub_dialogue imported successfully")
except ImportError as _e:
    print(f"pub_dialogue not found: {_e}")
    raise

This cell sets up the necessary API access for the language model and embedding model specified in the analysis, ensuring that the appropriate API keys are loaded from environment variables or user input. This step is crucial for enabling the subsequent clustering and analysis of concern and benefit phrase embeddings, as it establishes the connection to the models that will process the data. Without proper API configuration, the analysis would fail to execute, hindering the exploration of public dialogue dynamics.

In [ ]:
from pub_dialogue.client import LLMClient


In [ ]:
# @title Configure API access
import os as _os

# Load .env file if present (local development)
try:
    from dotenv import load_dotenv as _load_dotenv
    _load_dotenv()
except ImportError:
    pass

# Infer which API key env-var to prompt for from the model name prefix
_key_map = {"claude": "ANTHROPIC_API_KEY", "gemini": "GOOGLE_API_KEY"}
_key_var = next(
    (v for k, v in _key_map.items() if LLM_MODEL.startswith(k)),
    "OPENAI_API_KEY",
)

# Try Colab secrets first
try:
    from google.colab import userdata as _userdata
    _secret = _userdata.get(_key_var)
    if _secret:
        _os.environ[_key_var] = _secret
        print(f"{_key_var} loaded from Colab secrets")
except Exception:
    pass

# Interactive prompt fallback
if not _os.environ.get(_key_var):
    from getpass import getpass as _getpass
    _os.environ[_key_var] = _getpass(f"Enter {_key_var}: ")
    print(f"{_key_var} entered manually")

client = LLMClient(model=LLM_MODEL, embedding_model=EMBEDDING_MODEL)
print(f"LLMClient ready — model: {LLM_MODEL}, embeddings: {EMBEDDING_MODEL}")

This cell initializes the analysis by loading pre-processed data artifacts from a previous notebook, which includes various dataframes and embeddings related to concerns and benefits extracted from public dialogue. By avoiding redundant calls to the LLM extraction API, it ensures efficiency and consistency in the analysis, allowing subsequent clustering and robustness assessments to focus solely on the loaded data. This step is crucial as it sets the foundation for accurate clustering and labeling of phrases, which are essential for understanding public sentiment and concerns in the context of technology.

In [ ]:
# @title Load extraction artifacts
# Loads all outputs written by 01_processing.ipynb so this notebook never
# calls the LLM extraction/embedding API or re-processes PDFs.

import json, numpy as np, pandas as pd
from pathlib import Path
from openpyxl import load_workbook  # noqa: F401 (ensures openpyxl available)

_artifacts = access.load_artifacts(OUTPUT_FOLDER, CHECKPOINT_FOLDER)
chunks_df           = _artifacts["chunks_df"]
concerns_df         = _artifacts["concerns_df"]
benefits_df         = _artifacts["benefits_df"]
concern_embeddings  = _artifacts["concern_embeddings"]
benefit_embeddings  = _artifacts["benefit_embeddings"]
concern_ids         = _artifacts["concern_ids"]
benefit_ids         = _artifacts["benefit_ids"]

# extracted_concerns.csv and extracted_benefits.csv are saved before the
# technology_meta merge in 01_processing.ipynb, so we re-apply it if needed.
if "technology_meta" not in concerns_df.columns:
    _tech = chunks_df[["chunk_id", "technology_meta"]]
    concerns_df = concerns_df.merge(_tech, on="chunk_id", how="left")
    benefits_df = benefits_df.merge(_tech, on="chunk_id", how="left")

# Rebuild metadata_lookup needed by benefit merge cell
_meta_candidates = list(OUTPUT_FOLDER.glob("*.xlsx")) + list(Path(".").glob("*.xlsx"))
if _meta_candidates:
    metadata_df = pd.read_excel(_meta_candidates[0])
    metadata_lookup = metadata_df.set_index("filename").to_dict("index")
else:
    metadata_lookup = {}
    print("WARNING: metadata xlsx not found — metadata_lookup is empty")

TECHNOLOGY_CATEGORIES = sorted(chunks_df["technology_meta"].dropna().unique().tolist())

print(f"Chunks:   {len(chunks_df):,}")
print(f"Concerns: {len(concerns_df):,}  |  embeddings: {concern_embeddings.shape}")
print(f"Benefits: {len(benefits_df):,}  |  embeddings: {benefit_embeddings.shape}")

This cell performs clustering on normalized embeddings of concern phrases using the KMeans algorithm, assigning each phrase to one of the specified number of clusters. By generating soft membership scores that indicate the degree to which each phrase belongs to each cluster, this analysis enhances the understanding of how concerns are grouped, which is crucial for interpreting public dialogue and identifying prevalent themes in sentiment analysis. The results, including cluster assignments and centroids, are saved for further evaluation, supporting the overall robustness of the clustering approach in the context of concern and benefit phrase analysis.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize


In [ ]:
# @title Cluster concern phrase embeddings

show_status(f"Clustering into {N_CONCERN_CLUSTERS} concern groups...")

embeddings_normalized = normalize(concern_embeddings)

kmeans = KMeans(
    n_clusters=N_CONCERN_CLUSTERS,
    random_state=RANDOM_SEED,
    n_init=10,
    max_iter=300
)
concern_cluster_assignments = kmeans.fit_predict(embeddings_normalized)

centroids = kmeans.cluster_centers_
centroids_normalized = normalize(centroids)

assert len(concern_cluster_assignments) == len(concerns_df), (
    f"Cluster assignment length ({len(concern_cluster_assignments)}) does not match "
    f"concerns_df rows ({len(concerns_df)})."
)

# Add cluster assignment to concerns dataframe
concerns_df['cluster_id'] = concern_cluster_assignments

# SOFT MEMBERSHIP: cosine similarity to each centroid
show_status("Computing soft membership scores...")
soft_membership = cosine_similarity(embeddings_normalized, centroids_normalized)

print(f"\nClustering Results:")
print(f"  Clusters: {N_CONCERN_CLUSTERS}")
print(f"  Concern phrases: {len(concerns_df)}")
print(f"  Soft membership matrix: {soft_membership.shape}")

cluster_sizes = np.bincount(concern_cluster_assignments)
print(f"  Cluster sizes: min={cluster_sizes.min()}, max={cluster_sizes.max()}, median={np.median(cluster_sizes):.0f}")

np.save(CHECKPOINT_FOLDER / "soft_membership.npy", soft_membership)
np.save(CHECKPOINT_FOLDER / "cluster_centroids.npy", centroids_normalized)

# Save enriched concerns DataFrame (with cluster_id) for downstream notebooks
concerns_df.to_csv(OUTPUT_FOLDER / "extracted_concerns.csv", index=False)

show_complete("Clustering complete")


This cell analyzes the composition of clusters by calculating the entropy of technology distributions within each cluster, allowing for the classification of clusters as either cross-cutting or technology-specific. By quantifying the diversity of technologies represented in each cluster, this analysis provides insights into the breadth of concerns and benefits associated with different technologies, which is crucial for understanding public dialogue dynamics and informing policy decisions. The results are visualized and saved for further analysis, enhancing the interpretability of the clustering outcomes from the previous processing steps.

In [ ]:
from scipy.stats import entropy


In [ ]:
# @title Characterise clusters by technology distribution

show_status("Analysing cluster composition...")

# For each cluster, compute entropy of technology distribution
cluster_entropy = {}
cluster_tech_dist = {}

technologies = concerns_df['technology_meta'].unique()
n_techs = len(technologies)

for cluster_id in range(N_CONCERN_CLUSTERS):
    cluster_mask = concerns_df['cluster_id'] == cluster_id
    cluster_data = concerns_df[cluster_mask]

    if len(cluster_data) == 0:
        cluster_entropy[cluster_id] = 0
        cluster_tech_dist[cluster_id] = {}
        continue

    # Technology distribution
    tech_counts = cluster_data['technology_meta'].value_counts()
    tech_probs = tech_counts / tech_counts.sum()

    # Entropy (higher = more cross-cutting)
    cluster_entropy[cluster_id] = entropy(tech_probs)
    cluster_tech_dist[cluster_id] = tech_counts.to_dict()

# Normalize entropy to 0-1 scale (max entropy = log(n_techs))
max_entropy = np.log(n_techs)
cluster_entropy_norm = {k: v / max_entropy for k, v in cluster_entropy.items()}

# Classify clusters
CROSS_CUTTING_THRESHOLD = 0.5  # Entropy > 50% of max = cross-cutting

cross_cutting_clusters = [k for k, v in cluster_entropy_norm.items() if v >= CROSS_CUTTING_THRESHOLD]
tech_specific_clusters = [k for k, v in cluster_entropy_norm.items() if v < CROSS_CUTTING_THRESHOLD]

print(f"\nCluster Classification:")
print(f"  Cross-cutting clusters (entropy >= {CROSS_CUTTING_THRESHOLD}): {len(cross_cutting_clusters)}")
print(f"  Technology-specific clusters: {len(tech_specific_clusters)}")
print(f"  Ratio: {len(cross_cutting_clusters)/N_CONCERN_CLUSTERS*100:.1f}% cross-cutting")

# Visualise entropy distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of entropy
axes[0].hist(list(cluster_entropy_norm.values()), bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({CROSS_CUTTING_THRESHOLD})')
axes[0].set_xlabel('Normalized Entropy')
axes[0].set_ylabel('Number of Clusters')
axes[0].set_title('Cluster Entropy Distribution\n(Higher = More Cross-Cutting)')
axes[0].legend()

# Entropy vs cluster size
sizes = [cluster_sizes[i] for i in range(N_CONCERN_CLUSTERS)]
entropies = [cluster_entropy_norm[i] for i in range(N_CONCERN_CLUSTERS)]
colors = ['green' if e >= CROSS_CUTTING_THRESHOLD else 'orange' for e in entropies]
axes[1].scatter(sizes, entropies, c=colors, alpha=0.6)
axes[1].axhline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--')
axes[1].set_xlabel('Cluster Size')
axes[1].set_ylabel('Normalized Entropy')
axes[1].set_title('Cross-Cutting (green) vs Technology-Specific (orange)')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "cluster_entropy_analysis.png", dpi=150)
plt.show()

show_complete("Cluster composition analysis complete")

# Persist entropy dicts so analysis notebooks can load them without re-running clustering
with open(OUTPUT_FOLDER / "cluster_entropy.json", "w") as _f:
    json.dump({
        "raw":  {str(k): v for k, v in cluster_entropy.items()},
        "norm": {str(k): v for k, v in cluster_entropy_norm.items()},
        "cross_cutting": cross_cutting_clusters,
    }, _f)

This cell defines essential helper functions and mappings for analyzing cross-cutting clusters in the context of concern and benefit phrase embeddings. By establishing a threshold for entropy and creating a label mapping for clusters, it facilitates the interpretation of clustering results, which is crucial for understanding the nuances of public dialogue and ensuring that the analysis remains robust across different sensitivity levels. This groundwork supports the subsequent evaluation of how well the clusters align with human-defined labels, enhancing the overall validity of the clustering approach.

In [ ]:
# --- Define cross-cutting helpers for Section 6 (missing in v10-2) ---

# Use the same threshold already used in Section 3
CROSS_CUTTING_ENTROPY_THRESHOLD = CROSS_CUTTING_THRESHOLD  # already 0.5 in the notebook

# IDs of cross-cutting clusters
cross_cutting_cluster_ids = set(cross_cutting_clusters)

# Optional: map cluster_id -> human label if we have GPT labels (summary_df)
label_map = {}
if "summary_df" in globals() and {"cluster_id", "label"}.issubset(summary_df.columns):
    label_map = dict(zip(summary_df["cluster_id"], summary_df["label"]))

# Provide the dict Section 6.2 expects
cross_cutting_labels = {cid: label_map.get(cid, f"Cluster {cid}") for cid in cross_cutting_cluster_ids}


---
# SECTION 4: Cluster interpretation and labelling
---

In [ ]:
# @title Extract exemplar concern phrases per cluster

show_status("Extracting exemplar concerns...")

N_EXEMPLARS = 8
cluster_exemplars = {}

for cluster_id in range(N_CONCERN_CLUSTERS):
    cluster_mask = concerns_df['cluster_id'] == cluster_id
    cluster_concerns = concerns_df[cluster_mask]
    cluster_embs = embeddings_normalized[cluster_mask]

    if len(cluster_concerns) == 0:
        continue

    centroid = centroids_normalized[cluster_id]
    similarities = cosine_similarity(cluster_embs, centroid.reshape(1, -1)).flatten()
    top_indices = np.argsort(similarities)[-N_EXEMPLARS:][::-1]

    exemplars = []
    for idx in top_indices:
        row = cluster_concerns.iloc[idx]
        exemplars.append({
            'concern': row['concern'],
            'technology': row['technology'],
            'year': int(row['year']) if pd.notna(row['year']) else None,
            'similarity': float(similarities[idx])
        })

    # Cluster metadata
    tech_dist = cluster_concerns['technology'].value_counts().head(3).to_dict()

    cluster_exemplars[cluster_id] = {
        'size': len(cluster_concerns),
        'entropy': cluster_entropy_norm[cluster_id],
        'is_cross_cutting': cluster_id in cross_cutting_clusters,
        'top_technologies': tech_dist,
        'exemplars': exemplars
    }

with open(OUTPUT_FOLDER / "cluster_exemplars.json", 'w') as f:
    json.dump(cluster_exemplars, f, indent=2, default=str)

show_complete(f"Extracted exemplars for {len(cluster_exemplars)} clusters")

This cell is responsible for assigning descriptive labels to clusters identified in the previous analysis, ensuring that each cluster is meaningfully interpreted in the context of concern and benefit phrases. By checking for cached labels and regenerating them if necessary, it maintains efficiency while also calculating a success rate for the labeling process; this is crucial for validating the reliability of the labels, as they directly influence subsequent analyses, such as framing-lens evaluations. If the labeling success rate falls below a predefined threshold, it prompts further investigation, underscoring the importance of accurate cluster interpretation in understanding public dialogue dynamics.

In [ ]:
import time
from tqdm.notebook import tqdm


In [ ]:
# @title Assign descriptive labels to clusters

MIN_CLUSTER_LABEL_SUCCESS_RATE = 0.90

# Check for cached labels
labels_file = OUTPUT_FOLDER / "cluster_labels.json"

if labels_file.exists():
    print("Found cached cluster labels...")
    with open(labels_file) as f:
        cluster_labels_dict = json.load(f)
    if len(cluster_labels_dict) == N_CONCERN_CLUSTERS:
        print(f"Using cached labels for {len(cluster_labels_dict)} clusters")
    else:
        print("Cache incomplete - regenerating")
        cluster_labels_dict = None
else:
    cluster_labels_dict = None

if cluster_labels_dict is None:
    show_status(f"Generating labels for {len(cluster_exemplars)} clusters...")
    cluster_labels_dict = {}
    failed_count = 0

    for cluster_id in tqdm(cluster_exemplars.keys(), desc="Labeling"):
        data = cluster_exemplars[cluster_id]
        result = du.label_cluster(cluster_id, data["exemplars"], data["is_cross_cutting"],
                                  kind="concern", client=client)
        cluster_labels_dict[str(cluster_id)] = result
        if not result.get("success", False):
            failed_count += 1
        time.sleep(0.3)

    with open(labels_file, "w") as f:
        json.dump(cluster_labels_dict, f, indent=2)

    show_complete(f"Labeling complete: {len(cluster_labels_dict) - failed_count} successful, {failed_count} failed")

_label_success = sum(1 for v in cluster_labels_dict.values() if v.get("success", False))
_label_total = max(1, len(cluster_labels_dict))
_label_success_rate = _label_success / _label_total
print(f"Cluster-label success rate: {_label_success_rate:.1%} ({_label_success}/{_label_total})")
if _label_success_rate < MIN_CLUSTER_LABEL_SUCCESS_RATE:
    raise RuntimeError(
        f"Cluster-label success rate ({_label_success_rate:.1%}) below threshold "
        f"({MIN_CLUSTER_LABEL_SUCCESS_RATE:.0%}). Re-run cluster labelling or inspect failures "
        f"before continuing; downstream framing-lens analysis depends on these labels."
    )

# Create label list
cluster_labels_list = [cluster_labels_dict.get(str(i), {}).get("label", f"Cluster {i}")
                       for i in range(N_CONCERN_CLUSTERS)]

# Summary
print("\nTop 15 Clusters by Size:")
cluster_summary = []
for cluster_id, data in cluster_exemplars.items():
    label = cluster_labels_dict.get(str(cluster_id), {}).get("label", f"Cluster {cluster_id}")
    cluster_summary.append({
        "cluster_id": cluster_id,
        "label": label,
        "size": data["size"],
        "entropy": data["entropy"],
        "type": "Cross-cutting" if data["is_cross_cutting"] else "Tech-specific",
    })

summary_df = pd.DataFrame(cluster_summary).sort_values("size", ascending=False)
print(summary_df[["cluster_id", "label", "size", "type"]].head(15).to_string(index=False))
summary_df.to_csv(OUTPUT_FOLDER / "cluster_summary.csv", index=False)

---
# SECTION 5: Framing lens analysis
---

In [ ]:
# @title Identify framing lenses

show_status("Generating framing lens suggestions...")

# Prepare cluster summary for GPT
cluster_info = []
for cluster_id, data in cluster_exemplars.items():
    label = cluster_labels_dict.get(str(cluster_id), {}).get('label', f'Cluster {cluster_id}')
    description = cluster_labels_dict.get(str(cluster_id), {}).get('description', '')
    cluster_type = 'cross-cutting' if data['is_cross_cutting'] else 'tech-specific'
    cluster_info.append(f"{cluster_id}. {label} ({cluster_type}, n={data['size']}): {description}")

cluster_summary_text = "\n".join(cluster_info)

lens_prompt = f"""Analyze these {N_CONCERN_CLUSTERS} concern clusters from UK public dialogue reports.

Clusters:
{cluster_summary_text}

Group these clusters into 8-12 higher-level FRAMING LENSES that capture how publics frame their concerns.

For each lens provide:
1. Name (2-4 words)
2. Description (1 sentence)
3. List of cluster IDs that belong to this lens

A cluster can belong to multiple lenses if appropriate.
Ensure all clusters are assigned to at least one lens.

Return JSON:
{{"framing_lenses": [
  {{"name": "...", "description": "...", "suggested_clusters": [0, 1, 2...]}},
  ...
]}}"""

try:
    content = client.complete(
        messages=[
            {"role": "system", "content": "Expert in public engagement and discourse analysis. Return only valid JSON."},
            {"role": "user", "content": lens_prompt}
        ],
        max_completion_tokens=8000,
    )

    if '```' in content:
        parts = content.split('```')
        for part in parts:
            if part.startswith('json'):
                content = part[4:].strip()
                break
            elif part.strip().startswith('{'):
                content = part.strip()
                break

    suggested_lenses = json.loads(content)

    # Retry for any clusters the LLM missed
    assigned = set(cid for lens in suggested_lenses['framing_lenses'] for cid in lens['suggested_clusters'])
    uncovered_ids = sorted(set(range(N_CONCERN_CLUSTERS)) - assigned)
    if uncovered_ids:
        print(f"Initial call missed {len(uncovered_ids)} clusters — retrying for: {uncovered_ids[:20]}")
        existing_names = [l['name'] for l in suggested_lenses['framing_lenses']]
        uncovered_info = [line for line in cluster_info if int(line.split('.')[0]) in set(uncovered_ids)]
        retry_prompt = (
            f"Some concern clusters were not assigned to any framing lens.\n"
            f"Existing lens names: {existing_names}\n\n"
            f"Unassigned clusters:\n" + "\n".join(uncovered_info) + "\n\n"
            "For each unassigned cluster, add its ID to the most appropriate existing lens.\n"
            "Return JSON listing only lenses that need updating:\n"
            "{\"framing_lenses\": [{\"name\": \"<existing name>\", \"description\": \"...\", \"suggested_clusters\": [<new ids only>]}, ...]}"
        )
        retry_raw = client.complete(
            messages=[
                {"role": "system", "content": "Expert in public engagement analysis. Return only valid JSON."},
                {"role": "user", "content": retry_prompt}
            ],
            max_completion_tokens=4000,
        )
        if '```' in retry_raw:
            for part in retry_raw.split('```'):
                if part.startswith('json'): retry_raw = part[4:].strip(); break
                elif part.strip().startswith('{'): retry_raw = part.strip(); break
        retry_lenses = json.loads(retry_raw)
        existing_by_name = {l['name']: l for l in suggested_lenses['framing_lenses']}
        for rl in retry_lenses['framing_lenses']:
            if rl['name'] in existing_by_name:
                existing_by_name[rl['name']]['suggested_clusters'].extend(rl['suggested_clusters'])
            else:
                suggested_lenses['framing_lenses'].append(rl)

    # After retry, assign any still-uncovered clusters to the largest lens as fallback
    assigned = set(cid for lens in suggested_lenses['framing_lenses'] for cid in lens['suggested_clusters'])
    still_uncovered = sorted(set(range(N_CONCERN_CLUSTERS)) - assigned)
    if still_uncovered:
        fallback_lens = max(suggested_lenses['framing_lenses'], key=lambda l: len(l['suggested_clusters']))
        print(f"Assigning {len(still_uncovered)} remaining clusters to '{fallback_lens['name']}'")
        fallback_lens['suggested_clusters'].extend(still_uncovered)

    print("Suggested Framing Lenses:\n")
    for lens in suggested_lenses['framing_lenses']:
        print(f"  {lens['name']}")
        print(f"    {lens['description']}")
        print(f"    Clusters: {lens['suggested_clusters'][:8]}...\n")

    show_complete(f"Generated {len(suggested_lenses['framing_lenses'])} framing lenses")

except Exception as e:
    print(f"Error generating lenses: {e}")
    suggested_lenses = {'framing_lenses': []}


This cell maps identified concern clusters to their corresponding framing lenses, creating a structured representation that links clusters of concerns to specific narrative frameworks. By ensuring that all concern clusters are assigned to a lens, the analysis can maintain comprehensive coverage, which is crucial for understanding how different concerns are framed in public dialogue. The output not only facilitates subsequent analyses but also highlights any gaps in cluster coverage that may need to be addressed for a robust interpretation of the data.

In [ ]:
# @title Map concern clusters to framing lenses

FRAMING_LENS_MAPPINGS = {}
for lens in suggested_lenses['framing_lenses']:
    FRAMING_LENS_MAPPINGS[lens['name']] = {
        'description': lens['description'],
        'cluster_ids': lens['suggested_clusters']
    }

with open(OUTPUT_FOLDER / "framing_lens_mappings.json", 'w') as f:
    json.dump(FRAMING_LENS_MAPPINGS, f, indent=2)

# Map clusters to lenses
cluster_to_lens = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    for cluster_id in data['cluster_ids']:
        if cluster_id not in cluster_to_lens:
            cluster_to_lens[cluster_id] = []
        cluster_to_lens[cluster_id].append(lens_name)

all_cluster_ids = set(range(N_CONCERN_CLUSTERS))
covered_cluster_ids = set(cluster_to_lens.keys())
uncovered = all_cluster_ids - covered_cluster_ids
if uncovered:
    print("Uncovered cluster IDs:", sorted(uncovered)[:30])
    raise RuntimeError(
        f"{len(uncovered)} of {N_CONCERN_CLUSTERS} concern clusters are not assigned to any "
        f"framing lens. The lens-suggestion step did not honour the 'all clusters covered' "
        f"requirement. Re-run the framing-lens identification cell or assign the missing "
        f"clusters manually before continuing."
    )

print("Framing Lens Mappings:")
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    n_clusters = len(data['cluster_ids'])
    # Count concerns in this lens
    lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
    n_concerns = lens_mask.sum()
    print(f"  {lens_name}: {n_clusters} clusters, {n_concerns} concerns")


This cell calculates the prevalence of different framing lenses by counting the number of concern phrases associated with each lens based on predefined cluster mappings. By visualizing the distribution of these lenses, the analysis provides insights into how various concerns are framed in public dialogue, which is crucial for understanding the narrative dynamics and potential biases in discourse surrounding specific issues. The resulting bar chart serves as a valuable tool for identifying which framing lenses dominate the conversation, thereby informing further qualitative analysis and interpretation.

In [ ]:
# @title Compute framing lens prevalence

show_status("Computing framing lens prevalence...")

lens_prevalence = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
    lens_prevalence[lens_name] = lens_mask.sum()

lens_prev_df = pd.DataFrame([
    {'Framing Lens': k, 'Concerns': v}
    for k, v in lens_prevalence.items()
]).sort_values('Concerns', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(lens_prev_df['Framing Lens'], lens_prev_df['Concerns'], color='steelblue')
ax.set_xlabel('Number of Concern Phrases')
ax.set_title('Framing Lens Prevalence Across All Documents')

for bar, val in zip(bars, lens_prev_df['Concerns']):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "framing_lens_prevalence.png", dpi=150, bbox_inches='tight')
plt.show()

show_complete("Framing lens prevalence computed")

This cell generates a dendrogram to visualize the hierarchical structure of concern clusters derived from phrase embeddings, allowing for an examination of how well the clusters align with the predefined framing lenses assigned by the language model. By comparing the data-driven clustering with the LLM's lens assignments using the Adjusted Rand Index, the analysis assesses the robustness of the framing lens scheme, providing insights into the coherence of the clusters and the effectiveness of the model in capturing nuanced concerns in public dialogue. This step is crucial for validating the interpretability of the model's outputs and ensuring that the identified clusters meaningfully reflect the underlying discourse.

In [ ]:
# @title Hierarchical structure of concern clusters (dendrogram)
# Validates the framing-lens scheme by showing whether concern clusters
# that the LLM grouped into the same lens also cluster together in a
# data-driven hierarchy. Strong correspondence supports the lens scheme;
# divergence is a finding worth reporting.

import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

show_status("Computing concern-cluster dendrogram...")

# L2-normalise centroids (cosine distance via euclidean on normed vectors)
_norm = np.linalg.norm(centroids, axis=1, keepdims=True)
centroids_normed = centroids / np.clip(_norm, 1e-12, None)

# Pairwise cosine distance and average linkage
distances = pdist(centroids_normed, metric="cosine")
Z = linkage(distances, method="average")

# Build leaf labels from existing cluster labels
cluster_label_lookup = {
    int(k): v.get("label", f"Cluster {k}")
    for k, v in cluster_labels_dict.items()
}
leaf_labels = [cluster_label_lookup.get(i, f"Cluster {i}")
               for i in range(len(centroids_normed))]

# Build cluster->lens lookup
cluster_to_lens = {}
for lens_name, info in FRAMING_LENS_MAPPINGS.items():
    for cid in info.get("cluster_ids", []):
        cluster_to_lens[int(cid)] = lens_name

# Assign a colour per lens
import matplotlib.cm as cm
lens_names = list(FRAMING_LENS_MAPPINGS.keys())
lens_colours = {name: cm.tab20(i / max(len(lens_names), 1))
                for i, name in enumerate(lens_names)}

# Plot
fig, ax = plt.subplots(figsize=(11, max(8, len(leaf_labels) * 0.18)))
dd = dendrogram(
    Z,
    labels=leaf_labels,
    orientation="left",
    leaf_font_size=8,
    color_threshold=0,
    above_threshold_color="grey",
    ax=ax,
)
# Recolour leaf labels by lens
ylabels = ax.get_ymajorticklabels()
for label in ylabels:
    cid_label = label.get_text()
    matched_cid = next((cid for cid, name in cluster_label_lookup.items()
                        if name == cid_label), None)
    if matched_cid is not None:
        lens = cluster_to_lens.get(matched_cid, None)
        if lens is not None:
            label.set_color(lens_colours.get(lens, "black"))

ax.set_xlabel("Cosine distance between concern cluster centroids")
ax.set_title("Hierarchical structure of concern clusters\n(leaf colour = LLM-assigned framing lens)")

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=col, label=name) for name, col in lens_colours.items()]
ax.legend(handles=legend_handles, loc="lower right", fontsize=7, title="Framing lens")

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "concern_cluster_dendrogram.png", dpi=200, bbox_inches="tight")
plt.show()

# Quantitative comparison: ARI between LLM lens assignment and dendrogram cut
from sklearn.metrics import adjusted_rand_score

n_lenses = len(FRAMING_LENS_MAPPINGS)
data_driven_groups = fcluster(Z, t=n_lenses, criterion="maxclust")
lens_assignment = np.array([
    lens_names.index(cluster_to_lens.get(i, lens_names[0]))
    if cluster_to_lens.get(i) in lens_names else -1
    for i in range(len(centroids_normed))
])
mask = lens_assignment >= 0
ari = adjusted_rand_score(lens_assignment[mask], data_driven_groups[mask])
print(f"\nNumber of concern lenses (LLM-derived): {n_lenses}")
print(f"Adjusted Rand Index between lens assignment and dendrogram cut at "
      f"{n_lenses} groups: {ari:.3f}")
print("Higher = stronger correspondence between data-driven hierarchy and "
      "LLM-derived lens scheme.")

show_complete("Concern-cluster dendrogram complete.")

This cell performs clustering on normalized benefit phrase embeddings using the KMeans algorithm, assigning each phrase to a specific cluster based on its semantic similarity. By analyzing the resulting clusters and their centroids, the analysis provides insights into the framing of benefits in public dialogue, which is crucial for understanding how different benefit narratives are constructed and perceived in various contexts. Additionally, the computation of soft membership scores allows for a nuanced interpretation of how closely each phrase aligns with the identified clusters, enhancing the robustness of the overall analysis.

In [ ]:
# @title Cluster benefit phrase embeddings

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)

show_status(f"Clustering into {N_BENEFIT_CLUSTERS} benefit groups...")

benefit_embeddings_normalized = normalize(benefit_embeddings)

benefit_kmeans = KMeans(
    n_clusters=N_BENEFIT_CLUSTERS,
    random_state=RANDOM_SEED,
    n_init=10,
    max_iter=300
)
benefit_cluster_assignments = benefit_kmeans.fit_predict(benefit_embeddings_normalized)

benefit_centroids = benefit_kmeans.cluster_centers_
benefit_centroids_normalized = normalize(benefit_centroids)

assert len(benefit_cluster_assignments) == len(benefits_df), (
    f"Benefit cluster assignment length ({len(benefit_cluster_assignments)}) "
    f"does not match benefits_df rows ({len(benefits_df)})."
)

# Add cluster assignment to benefits dataframe
benefits_df["cluster_id"] = benefit_cluster_assignments

# SOFT MEMBERSHIP: cosine similarity to each centroid
show_status("Computing soft membership scores...")
benefit_soft_membership = cosine_similarity(benefit_embeddings_normalized, benefit_centroids_normalized)

print(f"\nClustering Results:")
print(f"  Clusters: {N_BENEFIT_CLUSTERS}")
print(f"  Benefit phrases: {len(benefits_df)}")
print(f"  Soft membership matrix: {benefit_soft_membership.shape}")

benefit_cluster_sizes = np.bincount(benefit_cluster_assignments)
print(f"  Cluster sizes: min={benefit_cluster_sizes.min()}, max={benefit_cluster_sizes.max()}, median={np.median(benefit_cluster_sizes):.0f}")

np.save(CHECKPOINT_FOLDER / "benefit_soft_membership.npy", benefit_soft_membership)
np.save(CHECKPOINT_FOLDER / "benefit_cluster_centroids.npy", benefit_centroids_normalized)

# Save enriched benefits DataFrame (with cluster_id) for downstream notebooks
benefits_df.to_csv(OUTPUT_FOLDER / "extracted_benefits.csv", index=False)

show_complete("Benefit clustering complete")


This cell analyzes the composition of benefit clusters by examining the distribution of technologies associated with each cluster. By calculating the entropy of technology distributions, it identifies clusters that are cross-cutting (involving multiple technologies) versus those that are technology-specific, which is crucial for understanding how different technologies frame public dialogue around benefits. The results, including visualizations of entropy distributions and cluster classifications, provide insights into the diversity of perspectives within the data, informing subsequent analyses and interpretations.

In [ ]:
# @title Characterise benefit clusters by technology distribution

from scipy.stats import entropy as scipy_entropy

show_status("Analysing benefit cluster composition...")

benefit_cluster_entropy = {}
benefit_cluster_tech_dist = {}

technologies = benefits_df['technology_meta'].unique()
n_techs = len(technologies)

for cluster_id in range(N_BENEFIT_CLUSTERS):
    cluster_mask = benefits_df['cluster_id'] == cluster_id
    cluster_data = benefits_df[cluster_mask]

    if len(cluster_data) == 0:
        benefit_cluster_entropy[cluster_id] = 0
        benefit_cluster_tech_dist[cluster_id] = {}
        continue

    # Technology distribution
    tech_counts = cluster_data['technology_meta'].value_counts()
    tech_probs = tech_counts / tech_counts.sum()

    # Entropy (higher = more cross-cutting)
    benefit_cluster_entropy[cluster_id] = scipy_entropy(tech_probs)
    benefit_cluster_tech_dist[cluster_id] = tech_counts.to_dict()

# Normalize entropy to 0-1 scale (max entropy = log(n_techs))
max_entropy = np.log(n_techs)
normalized_entropy_benefits = {k: v / max_entropy for k, v in benefit_cluster_entropy.items()}

# Classify clusters
CROSS_CUTTING_THRESHOLD = 0.5  # Entropy > 50% of max = cross-cutting

cross_cutting_clusters_benefits = [k for k, v in normalized_entropy_benefits.items() if v >= CROSS_CUTTING_THRESHOLD]
tech_specific_clusters_benefits = [k for k, v in normalized_entropy_benefits.items() if v < CROSS_CUTTING_THRESHOLD]

print(f"\nBenefit Cluster Classification:")
print(f"  Cross-cutting clusters (entropy >= {CROSS_CUTTING_THRESHOLD}): {len(cross_cutting_clusters_benefits)}")
print(f"  Technology-specific clusters: {len(tech_specific_clusters_benefits)}")
print(f"  Ratio: {len(cross_cutting_clusters_benefits)/N_BENEFIT_CLUSTERS*100:.1f}% cross-cutting")

# Visualise entropy distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of entropy
axes[0].hist(list(normalized_entropy_benefits.values()), bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({CROSS_CUTTING_THRESHOLD})')
axes[0].set_xlabel('Normalized Entropy')
axes[0].set_ylabel('Number of Clusters')
axes[0].set_title('Benefit Cluster Entropy Distribution\n(Higher = More Cross-Cutting)')
axes[0].legend()

# Entropy vs cluster size
sizes = [int((benefits_df["cluster_id"] == i).sum()) for i in range(N_BENEFIT_CLUSTERS)]
entropies = [normalized_entropy_benefits[i] for i in range(N_BENEFIT_CLUSTERS)]
colors = ['green' if e >= CROSS_CUTTING_THRESHOLD else 'orange' for e in entropies]
axes[1].scatter(sizes, entropies, c=colors, alpha=0.6)
axes[1].axhline(CROSS_CUTTING_THRESHOLD, color='red', linestyle='--')
axes[1].set_xlabel('Cluster Size')
axes[1].set_ylabel('Normalized Entropy')
axes[1].set_title('Cross-Cutting (green) vs Technology-Specific (orange)')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_cluster_entropy_analysis.png", dpi=150)
plt.show()

show_complete("Benefit cluster composition analysis complete")

# Persist benefit entropy dicts so analysis notebooks can load them without re-running clustering
with open(OUTPUT_FOLDER / "benefit_cluster_entropy.json", "w") as _f:
    json.dump({
        "raw":  {str(k): v for k, v in benefit_cluster_entropy.items()},
        "norm": {str(k): v for k, v in normalized_entropy_benefits.items()},
        "cross_cutting": cross_cutting_clusters_benefits,
    }, _f)


This code segment establishes a framework for labeling clusters identified in the benefits analysis by ensuring that necessary variables and data structures are in place. It checks for the existence of cross-cutting clusters and a summary DataFrame, then creates a mapping of cluster IDs to their corresponding labels, which is crucial for interpreting the results of the clustering analysis in the context of public dialogue. This labeling aids in understanding how different benefits are framed across various clusters, which is essential for evaluating the effectiveness of communication strategies in public discourse.

In [ ]:
# --- Define cross-cutting helpers for Benefits (Section 6) ---

CROSS_CUTTING_ENTROPY_THRESHOLD = globals().get("CROSS_CUTTING_THRESHOLD", 0.5)

if "cross_cutting_clusters_benefits" not in globals():
    raise NameError("cross_cutting_clusters_benefits not found. Run the benefit entropy/classification cell first.")

cross_cutting_cluster_ids_benefits = set(cross_cutting_clusters_benefits)

label_map = {}
if "benefit_summary_df" in globals() and {"cluster_id", "label"}.issubset(benefit_summary_df.columns):
    label_map = dict(zip(benefit_summary_df["cluster_id"], benefit_summary_df["label"]))

cross_cutting_labels_benefits = {
    cid: label_map.get(cid, f"Cluster {cid}") for cid in cross_cutting_cluster_ids_benefits
}


---
# SECTION 4B: Cluster interpretation and labelling (benefits)
---

In [ ]:
# @title Extract exemplar benefit phrases per cluster

show_status("Extracting exemplar benefits...")

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)
N_EXEMPLARS = 8
benefit_cluster_exemplars = {}

assert "cluster_id" in benefits_df.columns, "benefits_df has no cluster_id column. Run cell 58 first."
assert benefits_df["cluster_id"].max() < N_BENEFIT_CLUSTERS, (
    f"benefits_df.cluster_id contains values >= {N_BENEFIT_CLUSTERS}. "
    "This usually means stale cluster IDs from a different clustering run; re-run cell 58."
)

for cluster_id in range(N_BENEFIT_CLUSTERS):
    cluster_mask = benefits_df["cluster_id"] == cluster_id
    cluster_benefits = benefits_df[cluster_mask]
    cluster_embs = benefit_embeddings_normalized[cluster_mask]

    if len(cluster_benefits) == 0:
        continue

    centroid = benefit_centroids_normalized[cluster_id]
    similarities = cosine_similarity(cluster_embs, centroid.reshape(1, -1)).flatten()
    top_indices = np.argsort(similarities)[-N_EXEMPLARS:][::-1]

    exemplars = []
    for i in top_indices:
        row = cluster_benefits.iloc[i]
        exemplars.append({
            "benefit": row["benefit"],
            "technology": row["technology"],
            "year": int(row["year"]) if pd.notna(row["year"]) else None,
            "similarity": float(similarities[i])
        })

    tech_dist = cluster_benefits["technology"].value_counts().head(3).to_dict()

    benefit_cluster_exemplars[cluster_id] = {
        "size": len(cluster_benefits),
        "entropy": normalized_entropy_benefits[cluster_id],
        "is_cross_cutting": cluster_id in cross_cutting_clusters_benefits,
        "top_technologies": tech_dist,
        "exemplars": exemplars
    }

with open(OUTPUT_FOLDER / "benefit_cluster_exemplars.json", "w") as f:
    json.dump(benefit_cluster_exemplars, f, indent=2, default=str)

show_complete(f"Extracted exemplars for {len(benefit_cluster_exemplars)} clusters")


This cell is responsible for assigning descriptive labels to clusters identified in the benefit phrase embeddings, ensuring that each cluster is accurately characterized based on its exemplars. By evaluating the success rate of the labeling process and generating a summary of the clusters, it provides critical insights into the nature of the benefits discussed in public dialogue, which is essential for interpreting the results of the clustering analysis and informing subsequent research or policy recommendations. The success rate threshold ensures that only reliable labels are used, maintaining the integrity of the analysis.

In [ ]:
# @title Assign descriptive labels to benefit clusters

MIN_CLUSTER_LABEL_SUCCESS_RATE = 0.90

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)

if "benefit_cluster_exemplars" not in globals():
    raise NameError("benefit_cluster_exemplars not found. Run the exemplar extraction cell first.")

benefit_labels_file = OUTPUT_FOLDER / "benefit_cluster_labels.json"
benefit_cluster_labels_dict = None

if benefit_labels_file.exists():
    print("Found cached benefit cluster labels...")
    with open(benefit_labels_file) as f:
        benefit_cluster_labels_dict = json.load(f)
    if len(benefit_cluster_labels_dict) == N_BENEFIT_CLUSTERS:
        print(f"Using cached labels for {len(benefit_cluster_labels_dict)} clusters")
    else:
        print("Cache incomplete - regenerating")
        benefit_cluster_labels_dict = None

if benefit_cluster_labels_dict is None:
    show_status(f"Generating labels for {len(benefit_cluster_exemplars)} clusters...")
    benefit_cluster_labels_dict = {}
    failed_count = 0

    for cluster_id in tqdm(benefit_cluster_exemplars.keys(), desc="Labeling"):
        data = benefit_cluster_exemplars[cluster_id]
        result = du.label_cluster(cluster_id, data["exemplars"], data["is_cross_cutting"],
                                  kind="benefit", client=client)
        benefit_cluster_labels_dict[str(cluster_id)] = result
        if not result.get("success", False):
            failed_count += 1
        time.sleep(0.3)

    with open(benefit_labels_file, "w") as f:
        json.dump(benefit_cluster_labels_dict, f, indent=2)

    show_complete(f"Labeling complete: {len(benefit_cluster_labels_dict) - failed_count} successful, {failed_count} failed")

_label_success = sum(1 for v in benefit_cluster_labels_dict.values() if v.get("success", False))
_label_total = max(1, len(benefit_cluster_labels_dict))
_label_success_rate = _label_success / _label_total
print(f"Benefit cluster-label success rate: {_label_success_rate:.1%} ({_label_success}/{_label_total})")
if _label_success_rate < MIN_CLUSTER_LABEL_SUCCESS_RATE:
    raise RuntimeError(
        f"Benefit cluster-label success rate ({_label_success_rate:.1%}) below threshold "
        f"({MIN_CLUSTER_LABEL_SUCCESS_RATE:.0%})."
    )

benefit_cluster_labels_list = [
    benefit_cluster_labels_dict.get(str(i), {}).get("label", f"Cluster {i}")
    for i in range(N_BENEFIT_CLUSTERS)
]

print("\nTop 15 Benefit Clusters by Size:")
cluster_summary = []
for cluster_id, data in benefit_cluster_exemplars.items():
    label = benefit_cluster_labels_dict.get(str(cluster_id), {}).get("label", f"Cluster {cluster_id}")
    cluster_summary.append({
        "cluster_id": cluster_id,
        "label": label,
        "size": data["size"],
        "entropy": data["entropy"],
        "type": "Cross-cutting" if data["is_cross_cutting"] else "Tech-specific",
    })

benefit_summary_df = pd.DataFrame(cluster_summary).sort_values("size", ascending=False)
print(benefit_summary_df[["cluster_id", "label", "size", "type"]].head(15).to_string(index=False))
benefit_summary_df.to_csv(OUTPUT_FOLDER / "benefit_cluster_summary.csv", index=False)

---
# SECTION 5B: Framing lens analysis (benefits)
---

In [ ]:
# @title Identify benefit framing lenses

show_status("Generating benefit framing lens suggestions...")

N_BENEFIT_CLUSTERS = globals().get("N_BENEFIT_CLUSTERS", N_CONCERN_CLUSTERS)

# Prepare cluster summary for GPT
cluster_info = []
for cluster_id, data in benefit_cluster_exemplars.items():
    label = benefit_cluster_labels_dict.get(str(cluster_id), {}).get('label', f'Cluster {cluster_id}')
    description = benefit_cluster_labels_dict.get(str(cluster_id), {}).get('description', '')
    cluster_type = 'cross-cutting' if data['is_cross_cutting'] else 'tech-specific'
    cluster_info.append(f"{cluster_id}. {label} ({cluster_type}, n={data['size']}): {description}")

cluster_summary_text = "\n".join(cluster_info)

lens_prompt = f"""Analyze these {N_BENEFIT_CLUSTERS} benefit clusters from UK public dialogue reports.

Clusters:
{cluster_summary_text}

Group these clusters into 8-12 higher-level FRAMING LENSES that capture how publics frame their benefits.

For each lens provide:
1. Name (2-4 words)
2. Description (1 sentence)
3. List of cluster IDs that belong to this lens

A cluster can belong to multiple lenses if appropriate.
Ensure all clusters are assigned to at least one lens.

Return JSON:
{{"framing_lenses": [
  {{"name": "...", "description": "...", "suggested_clusters": [0, 1, 2...]}},
  ...
]}}"""

try:
    content = client.complete(
        messages=[
            {"role": "system", "content": "Expert in public engagement and discourse analysis. Return only valid JSON."},
            {"role": "user", "content": lens_prompt}
        ],
        max_completion_tokens=8000,
    )

    if '```' in content:
        parts = content.split('```')
        for part in parts:
            if part.startswith('json'):
                content = part[4:].strip()
                break
            elif part.strip().startswith('{'):
                content = part.strip()
                break

    suggested_benefit_lenses = json.loads(content)

    # Retry for any clusters the LLM missed
    assigned = set(cid for lens in suggested_benefit_lenses['framing_lenses'] for cid in lens['suggested_clusters'])
    uncovered_ids = sorted(set(range(N_BENEFIT_CLUSTERS)) - assigned)
    if uncovered_ids:
        print(f"Initial call missed {len(uncovered_ids)} benefit clusters — retrying for: {uncovered_ids[:20]}")
        existing_names = [l['name'] for l in suggested_benefit_lenses['framing_lenses']]
        uncovered_info = [line for line in cluster_info if int(line.split('.')[0]) in set(uncovered_ids)]
        retry_prompt = (
            f"Some benefit clusters were not assigned to any framing lens.\n"
            f"Existing lens names: {existing_names}\n\n"
            f"Unassigned clusters:\n" + "\n".join(uncovered_info) + "\n\n"
            "For each unassigned cluster, add its ID to the most appropriate existing lens.\n"
            "Return JSON listing only lenses that need updating:\n"
            "{\"framing_lenses\": [{\"name\": \"<existing name>\", \"description\": \"...\", \"suggested_clusters\": [<new ids only>]}, ...]}"
        )
        retry_raw = client.complete(
            messages=[
                {"role": "system", "content": "Expert in public engagement analysis. Return only valid JSON."},
                {"role": "user", "content": retry_prompt}
            ],
            max_completion_tokens=4000,
        )
        if '```' in retry_raw:
            for part in retry_raw.split('```'):
                if part.startswith('json'): retry_raw = part[4:].strip(); break
                elif part.strip().startswith('{'): retry_raw = part.strip(); break
        retry_lenses = json.loads(retry_raw)
        existing_by_name = {l['name']: l for l in suggested_benefit_lenses['framing_lenses']}
        for rl in retry_lenses['framing_lenses']:
            if rl['name'] in existing_by_name:
                existing_by_name[rl['name']]['suggested_clusters'].extend(rl['suggested_clusters'])
            else:
                suggested_benefit_lenses['framing_lenses'].append(rl)

    # After retry, assign any still-uncovered clusters to the largest lens as fallback
    assigned = set(cid for lens in suggested_benefit_lenses['framing_lenses'] for cid in lens['suggested_clusters'])
    still_uncovered = sorted(set(range(N_BENEFIT_CLUSTERS)) - assigned)
    if still_uncovered:
        fallback_lens = max(suggested_benefit_lenses['framing_lenses'], key=lambda l: len(l['suggested_clusters']))
        print(f"Assigning {len(still_uncovered)} remaining benefit clusters to '{fallback_lens['name']}'")
        fallback_lens['suggested_clusters'].extend(still_uncovered)

    print("Suggested Benefit Framing Lenses:\n")
    for lens in suggested_benefit_lenses['framing_lenses']:
        print(f"  {lens['name']}")
        print(f"    {lens['description']}")
        print(f"    Clusters: {lens['suggested_clusters'][:8]}...\n")

    show_complete(f"Generated {len(suggested_benefit_lenses['framing_lenses'])} benefit framing lenses")

except Exception as e:
    print(f"Error generating lenses: {e}")
    suggested_benefit_lenses = {'framing_lenses': []}


This cell maps identified benefit clusters to their corresponding framing lenses, facilitating an understanding of how different benefits are perceived within public discourse. By creating a structured JSON output, it ensures that each cluster is associated with a specific lens, which is crucial for analyzing the effectiveness of framing strategies in conveying benefits. The cell also checks for any uncovered clusters, highlighting potential gaps in the analysis that could affect the robustness of subsequent interpretations and applications in NLP and public dialogue research.

In [ ]:
# @title Map benefit clusters to framing lenses

BENEFIT_FRAMING_LENS_MAPPINGS = {}
for lens in suggested_benefit_lenses['framing_lenses']:
    BENEFIT_FRAMING_LENS_MAPPINGS[lens['name']] = {
        'description': lens['description'],
        'cluster_ids': lens['suggested_clusters']
    }

with open(OUTPUT_FOLDER / "benefit_framing_lens_mappings.json", 'w') as f:
    json.dump(BENEFIT_FRAMING_LENS_MAPPINGS, f, indent=2)

# Map clusters to lenses (benefit-specific)
benefit_cluster_to_lens = {}
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    for cluster_id in data['cluster_ids']:
        if cluster_id not in benefit_cluster_to_lens:
            benefit_cluster_to_lens[cluster_id] = []
        benefit_cluster_to_lens[cluster_id].append(lens_name)

all_cluster_ids = set(range(N_BENEFIT_CLUSTERS))
covered_cluster_ids = set(benefit_cluster_to_lens.keys())
uncovered = all_cluster_ids - covered_cluster_ids
if uncovered:
    print("Uncovered benefit cluster IDs:", sorted(uncovered)[:30])
    raise RuntimeError(
        f"{len(uncovered)} of {N_BENEFIT_CLUSTERS} benefit clusters are not assigned to any "
        f"framing lens. Re-run the framing-lens identification cell or assign manually."
    )

print("Benefit Framing Lens Mappings:")
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    n_clusters = len(data['cluster_ids'])
    lens_mask = benefits_df['cluster_id'].isin(data['cluster_ids'])
    n_benefits = lens_mask.sum()
    print(f"  {lens_name}: {n_clusters} clusters, {n_benefits} benefits")


This cell conducts a sensitivity analysis by evaluating the clustering outcomes for three different cluster counts (60, 75, and 90) on the benefit phrase embeddings. By comparing these results, the analysis assesses how stable and robust the clustering is across varying levels of granularity, which is crucial for understanding the consistency of thematic framing in public dialogue about benefits. This step is essential for ensuring that the identified clusters meaningfully represent the underlying data and can inform further research or applications effectively.

In [ ]:
# @title Compare results across alternative cluster counts (60, 75, 90)

show_status("Running sensitivity analysis for cluster counts...")

ks = [60, 75, 90]

for k in ks:
    print(f"\n=== Sensitivity run: k={k} ===")
    du.run_sensitivity(
        k, "concern", embeddings_normalized, concerns_df, OUTPUT_FOLDER,
        random_seed=RANDOM_SEED, framing_lens_mappings=FRAMING_LENS_MAPPINGS,
    )

show_complete("Sensitivity runs complete (see outputs folder)")

This cell conducts a sensitivity analysis to evaluate how varying the number of clusters (60, 75, and 90) affects the results of benefit phrase embeddings. By examining these different cluster counts, the analysis aims to identify the robustness of the clustering outcomes, ensuring that the insights drawn about public perceptions of benefits are reliable and not overly dependent on the chosen clustering parameters. This step is crucial for validating the framing lens analysis, as it helps to ascertain the consistency of the identified benefit themes across different clustering configurations.

In [ ]:
# @title Compare benefit results across alternative cluster counts (60, 75, 90)

show_status("Running sensitivity analysis for benefit cluster counts...")

ks = [60, 75, 90]

for k in ks:
    print(f"\n=== Sensitivity run: k={k} ===")
    du.run_sensitivity(
        k, "benefit", benefit_embeddings_normalized, benefits_df, OUTPUT_FOLDER,
        random_seed=RANDOM_SEED, framing_lens_mappings=BENEFIT_FRAMING_LENS_MAPPINGS,
    )

show_complete("Benefit sensitivity runs complete (see outputs folder)")

This cell verifies the presence of all expected output artifacts and checkpoint files generated during the analysis, ensuring that the previous processing steps were completed successfully. By confirming the existence of these files, it safeguards the integrity of the data and results, which is crucial for subsequent clustering and framing lens analyses of concern and benefit phrases in public dialogue. This step is essential for maintaining reproducibility and reliability in the overall analysis workflow.

In [ ]:
# @title Artifact manifest — verify all expected outputs were written
# This cell asserts that every artifact expected by the analysis notebooks
# exists on disk.  Run it at the end of 01_processing to confirm a complete run.

from pathlib import Path
_out  = OUTPUT_FOLDER
_ckpt = CHECKPOINT_FOLDER

_EXPECTED_OUTPUT = [
    "paragraph_chunks.csv",
    "paragraph_chunks_per_document.csv",
    "extracted_concerns.csv",
    "extracted_benefits.csv",
    "cluster_labels.json",
    "cluster_summary.csv",
    "cluster_exemplars.json",
    "cluster_entropy.json",
    "framing_lens_mappings.json",
    "benefit_cluster_labels.json",
    "benefit_cluster_summary.csv",
    "benefit_cluster_exemplars.json",
    "benefit_cluster_entropy.json",
    "benefit_framing_lens_mappings.json",
]

_EXPECTED_CHECKPOINT = [
    "concern_embeddings.npy",
    "concern_ids.json",
    "cluster_centroids.npy",
    "benefit_embeddings.npy",
    "benefit_ids.json",
    "benefit_cluster_centroids.npy",
]

_missing = []
for _name in _EXPECTED_OUTPUT:
    _p = _out / _name
    if _p.exists():
        print(f"  OK   {_name}")
    else:
        print(f"  MISS {_name}")
        _missing.append(str(_p))

for _name in _EXPECTED_CHECKPOINT:
    _p = _ckpt / _name
    if _p.exists():
        print(f"  OK   checkpoints/{_name}")
    else:
        print(f"  MISS checkpoints/{_name}")
        _missing.append(str(_p))

if _missing:
    raise RuntimeError(
        f"\n{len(_missing)} artifact(s) missing — check earlier cells:\n"
        + "\n".join(f"  {m}" for m in _missing)
    )
print(f"\nAll {len(_EXPECTED_OUTPUT) + len(_EXPECTED_CHECKPOINT)} artifacts present.")